# S6E10 — 13 Auxiliary features: expected value of each column

> **Pipeline position:** 09 `X_rich` → 12 recipe test → **13 aux features** → 14 RealMLP-rich + aux → 11 stack

| | |
|---|---|
| **Input** | `data/raw/train.csv`, `test.csv` (the 21 raw columns; **no labels**) |
| **Output** | `data/processed/X_aux_train.parquet`, `X_aux_test.parquet` (32 columns); per-column checkpoints in `data/processed/aux_parts/` |
| **Runtime** | ≈ 20–30 min on 12 CPU threads (80 LightGBM fits), once |
| **Labels used?** | **No.** Only the 21 input columns of train+test; the satisfaction label is never read. |

**Idea:** for 16 columns, a model learns to predict the column **from the other 20** (on train+test together). Its prediction is the **expected value** of that column for "a passenger like this one" (`aux_ev_*`); the difference between the actual and the expected value is the **deviation** (`aux_dev_*`). Example: two passengers rate `seat_comfort` 3; for one of them the context (class, distance, other ratings) predicts 4.2, for the other 2.1 — the first is unusually critical, the second unusually generous. No single column contains this; the network would have to learn the relation between 20 columns and the one rating on its own, from labels.

**Why it might help when rating summaries did not:** the summaries (mean / min / max of ratings) were tried and gave nothing. A summary is the *same* for every passenger with the same ratings; the expected value depends on the *context* (class, travel type, distance, age, the other ratings), so the deviation is new by construction. The idea is used in a public solution of this competition (the author of the TabPFN members uses `aux_ev_*` columns). **Realistic expectation:** 0 to +0.0002 on fold 0; the same null result as for the summaries is entirely possible.

**Cross-prediction (no self-leakage):** train+test (≈ 1M rows) are split into 5 parts; each part is predicted by a model trained on the other 4. So a row's value is never seen by the model that predicts it — otherwise the expected value would just copy the actual value and the deviation would be ~0. These 5 parts are unrelated to the competition folds; since no labels are involved, that is safe.

**Columns (16):** the 13 ratings (regression → expected rating) + `type_of_travel`, `customer_type`, `class` as P(Business) (binary → probability). For the binary ones the deviation is `indicator − probability`.

**Model:** a small LightGBM (300 trees, 63 leaves, lr 0.1). It captures non-linear relations between ratings; a linear model would only give a combination the network can compute itself; RealMLP would be slower and of the same family as the consumer.

> **Glossary:** *Auxiliary task* = a prediction task other than the target (here: predict one input column from the others), used to create features. *Cross-prediction* = every row is predicted by a model that did not see it. *R²* = share of a column's variance explained by the model (1 = perfect, 0 = no better than the mean).

In [1]:
import re, time, warnings
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, roc_auc_score
import lightgbm as lgb
warnings.filterwarnings('ignore')

SMOKE = False      # True: 50 trees on 100k rows, only checks that the code runs (results are not saved)

RAW = next(p for p in [Path('data/raw'), Path('../data/raw'), Path('/kaggle/input/playground-series-s6e10')] if (p / 'train.csv').exists())
PROC = next(p for p in [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working')] if (p / 'X_rich_train.parquet').exists())
PARTS = PROC / 'aux_parts'; PARTS.mkdir(exist_ok=True)

train = pd.read_csv(RAW / 'train.csv'); test = pd.read_csv(RAW / 'test.csv')
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [c for c in test.columns if c not in CATS + NUMS + ['id']]
BASE = NUMS + RATINGS + CATS
assert len(RATINGS) == 13 and len(BASE) == 21
# Leakage guard: the label column is dropped right here and never used below.
full = pd.concat([train[BASE], test[BASE]], ignore_index=True)
n_train = len(train)
for c in CATS:
    full[c] = full[c].astype('category')
clean = lambda s: re.sub(r'[^0-9a-zA-Z]+', '_', s).strip('_').lower()
print(full.shape, '| levels:', {c: list(full[c].cat.categories) for c in CATS})

(999479, 21) | levels: {'Gender': ['Female', 'Male'], 'Customer Type': ['Loyal Customer', 'disloyal Customer'], 'Type of Travel': ['Business travel', 'Personal Travel'], 'Class': ['Business', 'Eco', 'Eco Plus']}


## 1. Targets of the auxiliary models
Ratings are predicted as numbers (regression → expected rating 0–5). The three categorical columns become 0/1 indicators (binary → probability). `Arrival Delay` keeps its 292 missing values; LightGBM handles NaN natively.

In [2]:
TARGETS = {r: ('reg', full[r].astype(float).values) for r in RATINGS}
TARGETS['Type of Travel'] = ('bin', (full['Type of Travel'] == 'Business travel').astype(int).values)
TARGETS['Customer Type'] = ('bin', (full['Customer Type'] == 'Loyal Customer').astype(int).values)
TARGETS['Class'] = ('bin', (full['Class'] == 'Business').astype(int).values)
for c, (kind, v) in TARGETS.items(): assert not np.isnan(v).any(), c
print({c: (k, round(float(v.mean()), 3)) for c, (k, v) in TARGETS.items()})

PARAMS = dict(n_estimators=50 if SMOKE else 300, learning_rate=0.1, num_leaves=63, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, verbosity=-1, random_state=42)
# 5 parts over train+test (label-free, so unrelated to the competition folds); each part is predicted by a model that never saw it.
AUX_FOLDS = list(KFold(5, shuffle=True, random_state=42).split(np.zeros(len(full))))

{'Inflight wifi service': ('reg', 2.778), 'Departure/Arrival time convenient': ('reg', 3.155), 'Ease of Online booking': ('reg', 2.813), 'Gate location': ('reg', 3.014), 'Food and drink': ('reg', 3.267), 'Online boarding': ('reg', 3.365), 'Seat comfort': ('reg', 3.57), 'Inflight entertainment': ('reg', 3.455), 'On-board service': ('reg', 3.516), 'Leg room service': ('reg', 3.472), 'Baggage handling': ('reg', 3.805), 'Checkin service': ('reg', 3.421), 'Cleanliness': ('reg', 3.382), 'Type of Travel': ('bin', 0.711), 'Customer Type': ('bin', 0.825), 'Class': ('bin', 0.489)}


## 2. Cross-predictions, one column at a time (resumable)
Every finished column is saved to `data/processed/aux_parts/<column>.npy`; rerunning continues with the missing ones. The quality line per column (R² or AUC) shows how predictable the column is from the other 20 — a column that is barely predictable gives an expected value ≈ constant and a deviation ≈ the column itself (little new information).

In [3]:
rows = n_train if not SMOKE else 100_000
EV = {}
for col, (kind, t) in TARGETS.items():
    f = PARTS / f'{clean(col)}.npy'
    if f.exists() and not SMOKE:
        EV[col] = np.load(f); print(f'{col:35s} loaded'); continue
    t0 = time.time(); Xc = full.drop(columns=[col]); ev = np.full(len(full), np.nan)
    for a, b in AUX_FOLDS:
        if SMOKE: a, b = a[:rows], b[:rows // 4]
        m = (lgb.LGBMRegressor if kind == 'reg' else lgb.LGBMClassifier)(**PARAMS)
        m.fit(Xc.iloc[a], t[a])
        ev[b] = m.predict(Xc.iloc[b]) if kind == 'reg' else m.predict_proba(Xc.iloc[b])[:, 1]
    ok = ~np.isnan(ev)
    q = r2_score(t[ok], ev[ok]) if kind == 'reg' else roc_auc_score(t[ok], ev[ok])
    print(f'{col:35s} {"R2" if kind == "reg" else "AUC"} {q:.4f}  ({time.time() - t0:.0f}s)', flush=True)
    if not SMOKE:
        np.save(f, ev.astype('float32'))
    EV[col] = ev

Inflight wifi service               R2 0.9273  (27s)


Departure/Arrival time convenient   R2 0.7483  (28s)


Ease of Online booking              R2 0.9095  (26s)


Gate location                       R2 0.6500  (23s)


Food and drink                      R2 0.6303  (24s)


...

Checkin service                     R2 0.2805  (23s)


Cleanliness                         R2 0.7533  (23s)


Type of Travel                      AUC 0.9996  (36s)


Customer Type                       AUC 0.9993  (35s)


Class                               AUC 0.9924  (33s)


## 3. Assemble, check and save
`aux_ev_<col>` = expected value, `aux_dev_<col>` = actual − expected. Checks: no NaN, train and test distributions of every new column alike (they are produced by the same cross-prediction, so a gap would point to a bug).

In [4]:
if not SMOKE:
    A = pd.DataFrame(index=full.index)
    for col, (kind, t) in TARGETS.items():
        A[f'aux_ev_{clean(col)}'] = EV[col].astype('float32')
        A[f'aux_dev_{clean(col)}'] = (t - EV[col]).astype('float32')
    assert not A.isna().any().any() and A.shape[1] == 32
    Atr, Ate = A.iloc[:n_train].reset_index(drop=True), A.iloc[n_train:].reset_index(drop=True)
    gap = (Atr.mean() - Ate.mean()).abs() / A.std()
    print('largest train/test mean gap (in std units):', round(float(gap.max()), 4), 'for', gap.idxmax())
    Atr.to_parquet(PROC / 'X_aux_train.parquet'); Ate.to_parquet(PROC / 'X_aux_test.parquet')
    print('saved', Atr.shape, Ate.shape)
    print(A.describe().T[['mean', 'std', 'min', 'max']].round(3).to_string())

largest train/test mean gap (in std units): 0.0037 for aux_dev_leg_room_service


saved (699635, 32) (299844, 32)


                                            mean    std    min    max
aux_ev_inflight_wifi_service               2.778  1.230 -0.501  5.679
aux_dev_inflight_wifi_service             -0.000  0.345 -4.982  5.002
aux_ev_departure_arrival_time_convenient   3.155  1.267 -0.728  5.268
aux_dev_departure_arrival_time_convenient  0.000  0.740 -4.970  5.183
aux_ev_ease_of_online_booking              2.813  1.265 -1.391  5.728
aux_dev_ease_of_online_booking             0.000  0.400 -4.986  5.014
aux_ev_gate_location                       3.014  1.009  0.710  5.223
aux_dev_gate_location                      0.000  0.742 -3.910  3.945
aux_ev_food_and_drink                      3.267  1.030  0.586  5.545
aux_dev_food_and_drink                    -0.000  0.791 -4.542  3.838
aux_ev_online_boarding                     3.365  1.180 -0.639  5.896
aux_dev_online_boarding                    0.000  0.509 -5.047  5.173
aux_ev_seat_comfort                        3.570  1.133  0.518  5.544
aux_dev_seat_comfort

## Results & decisions
**Built:** 32 columns, ≈ 7 min on 12 CPU threads (25–35 s per column). Train/test means agree (largest gap 0.004 std).

| Column group | Predictability from the other 20 columns |
|---|---|
| inflight entertainment, wifi, online booking | R² 0.91–0.96 (almost determined by the others) |
| online boarding, seat comfort, cleanliness, time convenient | R² 0.75–0.84 |
| gate, food, baggage, on-board service | R² 0.54–0.65 |
| leg room, check-in | R² 0.28–0.39 (mostly independent) |
| travel type / customer type / class | AUC 0.9996 / 0.9993 / 0.9924 |

The synthetic data is strongly structured: the three trip categories are almost fully determined by the other columns.

**Usefulness (decided in `14`): none.** RealMLP-rich + all 32 columns on fold 0: 0.96138 vs 0.96138 paired reference (+0.000005). The features are kept on disk for reproducibility but are **not used** downstream; testing them in tree models as well was skipped, since they did not help the network (`14`).


## Glossary
* **Auxiliary task** — predicting an input column from the other columns; no target involved.
* **Expected value (`aux_ev_*`)** — the cross-predicted value of a column for this passenger's context.
* **Deviation (`aux_dev_*`)** — actual value minus expected value: how unusual this answer is for this kind of passenger.
* **Cross-prediction** — each row is predicted by a model trained on the other parts of the data, never on the row itself.
* **R²** — share of variance explained; **AUC** — ranking quality for the binary columns.